[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notes/notebooks/10-sql.ipynb)


# SQL y Python: conexiones a bases de datos y fuentes de información

## De archivos locales a bases SQL y servicios Cloud

### Objetivo

Aprender a obtener datos desde Python/Google Colab **sin depender únicamente de CSV**.

En este notebook veremos:

- CSV, Excel, JSON y Parquet;
- SQLite;
- DuckDB;
- MySQL / MariaDB;
- PostgreSQL;
- Microsoft SQL Server;
- Oracle;
- Google BigQuery;
- Google Cloud SQL;
- AWS RDS;
- Azure SQL Database;
- Snowflake;
- MongoDB como ejemplo NoSQL;
- SQLAlchemy como interfaz común;
- consultas `SELECT`, `WHERE`, `JOIN`, `GROUP BY`, `HAVING`, CTE y funciones de ventana;
- lectura y escritura con pandas;
- manejo seguro de credenciales.

> El objetivo es aprender **qué cambia en la conexión** y qué partes del flujo con SQL y pandas se mantienen.



# 1. Archivo, base de datos y servicio Cloud no son lo mismo

## Archivos

- CSV
- Excel
- JSON
- Parquet

Se leen desde disco, Drive, URL o almacenamiento cloud.

## Bases de datos

- SQLite
- MySQL
- PostgreSQL
- SQL Server
- Oracle

Permiten almacenar tablas y ejecutar consultas SQL.

## Plataformas Cloud

- BigQuery
- Cloud SQL
- AWS RDS
- Azure SQL
- Snowflake

Pueden administrar motores tradicionales o funcionar como plataformas analíticas.



# 2. Datos necesarios para conectarse a una base remota

Normalmente necesitamos:

| Parámetro | Ejemplo |
|---|---|
| Motor | PostgreSQL |
| Host | `servidor.empresa.com` |
| Puerto | `5432` |
| Base de datos | `ventas` |
| Usuario | `analista` |
| Contraseña | `********` |
| Driver | `psycopg2` |

Puertos frecuentes:

| Motor | Puerto típico |
|---|---:|
| MySQL | 3306 |
| PostgreSQL | 5432 |
| SQL Server | 1433 |
| Oracle | 1521 |



# 3. Seguridad de credenciales

No es recomendable escribir contraseñas directamente:

```python
password = "MiPassword123"
```

Puede terminar accidentalmente en GitHub o en un notebook compartido.

Usaremos:

1. **Google Colab Secrets**
2. variables de entorno

En proyectos empresariales también pueden utilizarse Secret Managers.


In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def get_secret(nombre, default=None):
    """
    Busca primero en Google Colab Secrets.
    Si no existe, busca una variable de entorno.
    """
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor is not None:
            return valor
    except Exception:
        pass

    return os.getenv(nombre, default)

print("Entorno preparado.")



# 4. Librerías necesarias según el motor

No es necesario instalar todas.

En Colab puedes activar las que necesites.


In [ ]:

# Descomenta según el motor que necesites.

# %pip install -q sqlalchemy pymysql
# %pip install -q psycopg2-binary
# %pip install -q duckdb
# %pip install -q pyodbc
# %pip install -q oracledb
# %pip install -q google-cloud-bigquery pandas-gbq
# %pip install -q "cloud-sql-python-connector[pymysql]"
# %pip install -q snowflake-sqlalchemy
# %pip install -q pymongo


# 5. CSV: archivo de texto tabular

In [ ]:

# Ejemplo:
# df_csv = pd.read_csv("/content/ventas.csv")

# Desde una URL pública:
# df_csv = pd.read_csv("https://servidor.com/ventas.csv")


# 6. Excel

In [ ]:

# df_excel = pd.read_excel(
#     "/content/ventas.xlsx",
#     sheet_name="Hoja1"
# )

# Todas las hojas:
# hojas = pd.read_excel(
#     "/content/ventas.xlsx",
#     sheet_name=None
# )


# 7. JSON

In [ ]:

# df_json = pd.read_json(
#     "/content/clientes.json"
# )

# Para JSON anidado puede ser útil:
# pd.json_normalize(...)



# 8. Parquet

Parquet es muy usado en Data Engineering porque es:

- columnar;
- comprimido;
- eficiente;
- adecuado para grandes volúmenes.


In [ ]:

# df_parquet = pd.read_parquet(
#     "/content/ventas.parquet"
# )



# 9. SQLite: base SQL local sin servidor

SQLite viene con Python.

Vamos a crear una base completamente funcional con dos tablas:

- `CLIENTES`
- `FACTURAS`

Esto conserva la misma idea de una base relacional típica de ventas.


In [ ]:

import sqlite3

clientes = pd.DataFrame({
    "codigo_cliente": [1, 2, 3, 4, 5, 6],
    "nombre_cliente": [
        "Ana", "Carlos", "Laura",
        "Miguel", "Sofía", "Daniel"
    ],
    "edad": [28, 42, 35, 51, 26, 39],
    "codigo_forma_pago": [
        "TARJETA", "EFECTIVO", "PSE",
        "TARJETA", "PSE", "EFECTIVO"
    ],
    "descuento": [0.05, 0.00, 0.10, 0.05, 0.15, 0.00]
})

facturas = pd.DataFrame({
    "codigo_factura": [101,102,103,104,105,106,107,108,109,110],
    "codigo_cliente": [1,2,1,3,4,5,6,3,5,4],
    "venta": [850000,420000,1260000,730000,990000,540000,310000,1180000,870000,1500000],
    "costo": [510000,270000,760000,420000,610000,320000,190000,690000,510000,920000]
})

display(clientes)
display(facturas)


In [ ]:

conexion_sqlite = sqlite3.connect(
    "/content/ventas_demo.db"
)

clientes.to_sql(
    "CLIENTES",
    conexion_sqlite,
    if_exists="replace",
    index=False
)

facturas.to_sql(
    "FACTURAS",
    conexion_sqlite,
    if_exists="replace",
    index=False
)

print("Base SQLite creada correctamente.")


# 10. SELECT

In [ ]:

query = """
SELECT *
FROM CLIENTES
"""

df_clientes = pd.read_sql(
    query,
    conexion_sqlite
)

display(df_clientes)


# 11. WHERE + ORDER BY

In [ ]:

query = """
SELECT
    nombre_cliente,
    edad,
    descuento
FROM CLIENTES
WHERE edad >= 35
ORDER BY edad DESC
"""

display(
    pd.read_sql(
        query,
        conexion_sqlite
    )
)


# 12. INNER JOIN entre CLIENTES y FACTURAS

In [ ]:

query = """
SELECT
    C.nombre_cliente,
    C.edad,
    C.codigo_forma_pago,
    C.descuento,
    F.codigo_factura,
    F.venta,
    F.costo,
    F.venta - F.costo AS utilidad
FROM CLIENTES AS C
INNER JOIN FACTURAS AS F
    ON C.codigo_cliente = F.codigo_cliente
ORDER BY F.venta DESC
"""

df_ventas = pd.read_sql(
    query,
    conexion_sqlite
)

display(df_ventas)


# 13. GROUP BY

In [ ]:

query = """
SELECT
    C.codigo_forma_pago,
    COUNT(*) AS numero_facturas,
    SUM(F.venta) AS ventas_totales,
    AVG(F.venta) AS venta_promedio
FROM CLIENTES AS C
INNER JOIN FACTURAS AS F
    ON C.codigo_cliente = F.codigo_cliente
GROUP BY C.codigo_forma_pago
ORDER BY ventas_totales DESC
"""

display(
    pd.read_sql(
        query,
        conexion_sqlite
    )
)


# 14. HAVING

In [ ]:

query = """
SELECT
    C.codigo_forma_pago,
    SUM(F.venta) AS ventas_totales
FROM CLIENTES AS C
INNER JOIN FACTURAS AS F
    ON C.codigo_cliente = F.codigo_cliente
GROUP BY C.codigo_forma_pago
HAVING SUM(F.venta) > 2000000
ORDER BY ventas_totales DESC
"""

display(
    pd.read_sql(
        query,
        conexion_sqlite
    )
)


# 15. CTE — Common Table Expression

In [ ]:

query = """
WITH resumen AS (
    SELECT
        codigo_cliente,
        COUNT(*) AS compras,
        SUM(venta) AS venta_total
    FROM FACTURAS
    GROUP BY codigo_cliente
)
SELECT
    C.nombre_cliente,
    R.compras,
    R.venta_total
FROM resumen AS R
INNER JOIN CLIENTES AS C
    ON R.codigo_cliente = C.codigo_cliente
ORDER BY R.venta_total DESC
"""

display(
    pd.read_sql(
        query,
        conexion_sqlite
    )
)


# 16. Función de ventana

In [ ]:

query = """
SELECT
    codigo_factura,
    codigo_cliente,
    venta,
    RANK() OVER (
        ORDER BY venta DESC
    ) AS ranking_venta
FROM FACTURAS
ORDER BY ranking_venta
"""

display(
    pd.read_sql(
        query,
        conexion_sqlite
    )
)


# 17. Escribir un DataFrame en SQL con `to_sql()`

In [ ]:

productos = pd.DataFrame({
    "codigo_producto": [1, 2, 3],
    "producto": ["Producto A", "Producto B", "Producto C"],
    "precio": [120000, 85000, 190000]
})

productos.to_sql(
    "PRODUCTOS",
    conexion_sqlite,
    if_exists="replace",
    index=False
)

display(
    pd.read_sql(
        "SELECT * FROM PRODUCTOS",
        conexion_sqlite
    )
)



# 18. SQLAlchemy: una interfaz común

SQLAlchemy permite trabajar con diferentes motores de forma relativamente uniforme.

La conexión cambia, pero después podemos seguir usando:

```python
pd.read_sql(query, con=engine)
```


In [ ]:

from sqlalchemy import create_engine, URL, text

engine_sqlite = create_engine(
    "sqlite:////content/ventas_demo.db"
)

df_sqlalchemy = pd.read_sql(
    """
    SELECT
        C.nombre_cliente,
        F.venta,
        F.costo
    FROM CLIENTES AS C
    INNER JOIN FACTURAS AS F
        ON C.codigo_cliente = F.codigo_cliente
    """,
    con=engine_sqlite
)

display(df_sqlalchemy)



# 19. Patrón general con SQLAlchemy

```python
url = URL.create(
    drivername="motor+driver",
    username=usuario,
    password=password,
    host=host,
    port=puerto,
    database=base
)

engine = create_engine(url)
```

`URL.create()` es preferible a concatenar manualmente usuario y contraseña.


# 20. MySQL / MariaDB

In [ ]:

EJECUTAR_MYSQL = False

if EJECUTAR_MYSQL:

    usuario = get_secret("MYSQL_USER")
    password = get_secret("MYSQL_PASSWORD")
    host = get_secret("MYSQL_HOST")
    base = get_secret("MYSQL_DATABASE")
    puerto = int(get_secret("MYSQL_PORT", "3306"))

    if not all([usuario, password, host, base]):
        raise ValueError("Faltan secretos MYSQL_*.")

    url_mysql = URL.create(
        drivername="mysql+pymysql",
        username=usuario,
        password=password,
        host=host,
        port=puerto,
        database=base
    )

    engine_mysql = create_engine(url_mysql)

    df_mysql = pd.read_sql(
        "SELECT * FROM CLIENTES LIMIT 10",
        con=engine_mysql
    )

    display(df_mysql)


## JOIN en MySQL

In [ ]:

# Una vez exista engine_mysql:
#
# query = """
# SELECT
#     C.nombre_cliente,
#     F.venta,
#     F.costo,
#     C.edad,
#     C.codigo_forma_pago,
#     C.descuento
# FROM CLIENTES AS C
# INNER JOIN FACTURAS AS F
#     ON F.codigo_cliente = C.codigo_cliente
# """
#
# df = pd.read_sql(query, con=engine_mysql)
# display(df.head())


# 21. PostgreSQL

In [ ]:

EJECUTAR_POSTGRESQL = False

if EJECUTAR_POSTGRESQL:

    usuario = get_secret("POSTGRES_USER")
    password = get_secret("POSTGRES_PASSWORD")
    host = get_secret("POSTGRES_HOST")
    base = get_secret("POSTGRES_DATABASE")
    puerto = int(get_secret("POSTGRES_PORT", "5432"))

    url_postgres = URL.create(
        drivername="postgresql+psycopg2",
        username=usuario,
        password=password,
        host=host,
        port=puerto,
        database=base
    )

    engine_postgres = create_engine(url_postgres)

    df_postgres = pd.read_sql(
        "SELECT * FROM clientes LIMIT 10",
        con=engine_postgres
    )

    display(df_postgres)



# 22. Microsoft SQL Server

Driver frecuente:

`pyodbc`

> Además de instalar `pyodbc`, el sistema necesita un **ODBC Driver for SQL Server**.


In [ ]:

EJECUTAR_SQLSERVER = False

if EJECUTAR_SQLSERVER:

    import urllib.parse

    servidor = get_secret("SQLSERVER_HOST")
    base = get_secret("SQLSERVER_DATABASE")
    usuario = get_secret("SQLSERVER_USER")
    password = get_secret("SQLSERVER_PASSWORD")

    driver = "ODBC Driver 18 for SQL Server"

    odbc_string = (
        f"DRIVER={{{driver}}};"
        f"SERVER={servidor};"
        f"DATABASE={base};"
        f"UID={usuario};"
        f"PWD={password};"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
    )

    params = urllib.parse.quote_plus(
        odbc_string
    )

    engine_sqlserver = create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}"
    )

    df_sqlserver = pd.read_sql(
        "SELECT TOP 10 * FROM CLIENTES",
        con=engine_sqlserver
    )

    display(df_sqlserver)


# 23. Oracle Database

In [ ]:

EJECUTAR_ORACLE = False

if EJECUTAR_ORACLE:

    usuario = get_secret("ORACLE_USER")
    password = get_secret("ORACLE_PASSWORD")
    host = get_secret("ORACLE_HOST")
    service_name = get_secret("ORACLE_SERVICE_NAME")
    puerto = int(get_secret("ORACLE_PORT", "1521"))

    url_oracle = URL.create(
        drivername="oracle+oracledb",
        username=usuario,
        password=password,
        host=host,
        port=puerto,
        query={
            "service_name": service_name
        }
    )

    engine_oracle = create_engine(
        url_oracle
    )

    df_oracle = pd.read_sql(
        "SELECT * FROM CLIENTES FETCH FIRST 10 ROWS ONLY",
        con=engine_oracle
    )

    display(df_oracle)



# 24. DuckDB

DuckDB es una base analítica embebida.

Muy útil para:

- Parquet;
- archivos grandes;
- análisis local;
- SQL sin servidor.


In [ ]:

try:
    import duckdb

    con_duck = duckdb.connect()

    con_duck.register(
        "clientes_df",
        clientes
    )

    con_duck.register(
        "facturas_df",
        facturas
    )

    resultado_duck = con_duck.execute(
        """
        SELECT
            C.nombre_cliente,
            SUM(F.venta) AS ventas_totales
        FROM clientes_df AS C
        INNER JOIN facturas_df AS F
            ON C.codigo_cliente = F.codigo_cliente
        GROUP BY C.nombre_cliente
        ORDER BY ventas_totales DESC
        """
    ).df()

    display(resultado_duck)

except ImportError:
    print("Instala DuckDB con: %pip install -q duckdb")


## DuckDB puede consultar archivos directamente

In [ ]:

# CSV:
# con_duck.execute("""
#     SELECT *
#     FROM read_csv_auto('/content/ventas.csv')
#     LIMIT 10
# """).df()

# Parquet:
# con_duck.execute("""
#     SELECT *
#     FROM read_parquet('/content/ventas.parquet')
#     LIMIT 10
# """).df()



# 25. Google BigQuery

BigQuery es un data warehouse analítico serverless.

En Google Colab puede autenticarse una cuenta Google y ejecutar SQL directamente.


In [ ]:

EJECUTAR_BIGQUERY = False

if EJECUTAR_BIGQUERY:

    from google.colab import auth
    from google.cloud import bigquery

    auth.authenticate_user()

    project_id = get_secret(
        "GCP_PROJECT_ID"
    )

    client = bigquery.Client(
        project=project_id
    )

    query = """
    SELECT *
    FROM `proyecto.dataset.tabla`
    LIMIT 10
    """

    df_bigquery = (
        client
        .query(query)
        .to_dataframe()
    )

    display(df_bigquery)



# 26. Google Cloud SQL

Cloud SQL administra motores como:

- MySQL
- PostgreSQL
- SQL Server

Para MySQL/PostgreSQL puede utilizarse el Cloud SQL Python Connector.


In [ ]:

EJECUTAR_CLOUD_SQL_MYSQL = False

if EJECUTAR_CLOUD_SQL_MYSQL:

    from google.cloud.sql.connector import Connector

    connector = Connector()

    instance_connection_name = get_secret(
        "CLOUD_SQL_INSTANCE"
    )

    usuario = get_secret("CLOUD_SQL_USER")
    password = get_secret("CLOUD_SQL_PASSWORD")
    base = get_secret("CLOUD_SQL_DATABASE")

    def getconn():
        return connector.connect(
            instance_connection_name,
            "pymysql",
            user=usuario,
            password=password,
            db=base
        )

    engine_cloud_sql = create_engine(
        "mysql+pymysql://",
        creator=getconn
    )

    df_cloud = pd.read_sql(
        "SELECT * FROM CLIENTES LIMIT 10",
        con=engine_cloud_sql
    )

    display(df_cloud)



# 27. AWS RDS

RDS administra motores como:

- MySQL;
- PostgreSQL;
- MariaDB;
- SQL Server;
- Oracle.

La conexión desde Python se hace con el **driver del motor correspondiente**.

Ejemplo conceptual para PostgreSQL:

```python
url = URL.create(
    drivername="postgresql+psycopg2",
    username=usuario,
    password=password,
    host="endpoint-de-rds.amazonaws.com",
    port=5432,
    database="ventas"
)
```

La principal diferencia es el `host`: será el endpoint de RDS.



# 28. Azure SQL Database

Azure SQL usa tecnología SQL Server.

Por tanto, desde Python puede conectarse con:

- `pyodbc`;
- SQLAlchemy;
- ODBC Driver for SQL Server.

El servidor suele tener una forma similar a:

`mi-servidor.database.windows.net`


# 29. Snowflake

In [ ]:

EJECUTAR_SNOWFLAKE = False

if EJECUTAR_SNOWFLAKE:

    from snowflake.sqlalchemy import URL as SnowflakeURL

    engine_snowflake = create_engine(
        SnowflakeURL(
            account=get_secret("SNOWFLAKE_ACCOUNT"),
            user=get_secret("SNOWFLAKE_USER"),
            password=get_secret("SNOWFLAKE_PASSWORD"),
            database=get_secret("SNOWFLAKE_DATABASE"),
            schema=get_secret("SNOWFLAKE_SCHEMA"),
            warehouse=get_secret("SNOWFLAKE_WAREHOUSE")
        )
    )

    df_snowflake = pd.read_sql(
        "SELECT * FROM CLIENTES LIMIT 10",
        con=engine_snowflake
    )

    display(df_snowflake)



# 30. Bonus: MongoDB — NoSQL

MongoDB no utiliza el modelo relacional SQL tradicional.

Almacena documentos similares a JSON.


In [ ]:

EJECUTAR_MONGODB = False

if EJECUTAR_MONGODB:

    from pymongo import MongoClient

    uri = get_secret("MONGODB_URI")

    cliente_mongo = MongoClient(
        uri
    )

    db = cliente_mongo["ventas"]
    coleccion = db["clientes"]

    documentos = list(
        coleccion
        .find({})
        .limit(10)
    )

    df_mongo = pd.DataFrame(
        documentos
    )

    display(df_mongo)



# 31. Comparación rápida

| Fuente / motor | SQL | Servidor | Python |
|---|---|---|---|
| CSV | No | No | `pd.read_csv()` |
| Excel | No | No | `pd.read_excel()` |
| JSON | No | No | `pd.read_json()` |
| Parquet | No | No | `pd.read_parquet()` |
| SQLite | Sí | No | `sqlite3`, SQLAlchemy |
| DuckDB | Sí | No | `duckdb` |
| MySQL | Sí | Sí | `pymysql`, SQLAlchemy |
| PostgreSQL | Sí | Sí | `psycopg2`, SQLAlchemy |
| SQL Server | Sí | Sí | `pyodbc`, SQLAlchemy |
| Oracle | Sí | Sí | `oracledb`, SQLAlchemy |
| BigQuery | Sí | Cloud | `google-cloud-bigquery` |
| Cloud SQL | Sí | Cloud | Connector / SQLAlchemy |
| Snowflake | Sí | Cloud | Snowflake Connector |
| MongoDB | NoSQL | Sí/Cloud | `pymongo` |



# 32. `read_sql_query`, `read_sql_table` y `read_sql`

## Consulta SQL

```python
pd.read_sql_query(
    "SELECT * FROM CLIENTES",
    con=engine
)
```

## Tabla completa

```python
pd.read_sql_table(
    "CLIENTES",
    con=engine
)
```

## Interfaz general

```python
pd.read_sql(
    query,
    con=engine
)
```


# 33. Consultas parametrizadas

In [ ]:

from sqlalchemy import text

edad_minima = 30

query = text("""
SELECT
    nombre_cliente,
    edad
FROM CLIENTES
WHERE edad > :edad_minima
""")

with engine_sqlite.connect() as conn:

    df_parametros = pd.read_sql(
        query,
        conn,
        params={
            "edad_minima": edad_minima
        }
    )

display(df_parametros)



Las consultas parametrizadas son preferibles a concatenar valores manualmente.

Ayudan a reducir riesgos como **SQL Injection**.


# 34. Transacciones

In [ ]:

with engine_sqlite.begin() as conn:

    conn.execute(
        text("""
        UPDATE CLIENTES
        SET descuento = :nuevo_descuento
        WHERE codigo_cliente = :codigo
        """),
        {
            "nuevo_descuento": 0.08,
            "codigo": 1
        }
    )

display(
    pd.read_sql(
        """
        SELECT *
        FROM CLIENTES
        WHERE codigo_cliente = 1
        """,
        con=engine_sqlite
    )
)



# 35. `to_sql()` y `if_exists`

```python
df.to_sql(
    "NOMBRE_TABLA",
    con=engine,
    if_exists="append",
    index=False
)
```

Opciones:

- `fail`: error si existe;
- `replace`: elimina y recrea;
- `append`: agrega registros.

> `replace` debe usarse con mucho cuidado en producción.


# 36. Leer tablas muy grandes por bloques

In [ ]:

# for bloque in pd.read_sql(
#     "SELECT * FROM FACTURAS",
#     con=engine_mysql,
#     chunksize=10000
# ):
#     print(bloque.shape)



# 37. Conexión directa vs SQLAlchemy

## Directa

- `sqlite3.connect()`
- `pymysql.connect()`
- `psycopg2.connect()`
- `pyodbc.connect()`

## SQLAlchemy

Ventajas:

- interfaz más uniforme;
- buena integración con pandas;
- transacciones;
- pooling de conexiones;
- cambio de motor más sencillo.


# 38. Probar conexión antes de consultas grandes

In [ ]:

with engine_sqlite.connect() as conn:

    resultado = conn.execute(
        text("SELECT 1")
    )

    print(
        "Resultado:",
        resultado.scalar()
    )

print("Conexión funcionando.")



# 39. Errores de conexión frecuentes

## `ModuleNotFoundError`
Falta instalar el driver.

## `Access denied`
Usuario o contraseña incorrectos.

## `Connection refused`
Host/puerto incorrecto o servidor inaccesible.

## Timeout
Firewall, VPN o reglas de red.

## SSL error
La base exige configuración TLS/SSL.

## Unknown database
Nombre de base incorrecto.

## ODBC driver not found
Falta el driver del sistema para SQL Server.



# 40. Flujo recomendado

**Identificar fuente → instalar driver → guardar secretos → construir conexión → probar `SELECT 1` → consultar pocas filas → validar tipos → construir SQL → llevar resultado a pandas → analizar/modelar**



# 41. ¿Qué tecnología utilizar?

## Archivo pequeño
CSV / Excel.

## Datos analíticos en archivo
Parquet.

## Base local
SQLite.

## SQL analítico local
DuckDB.

## Aplicación web
MySQL / PostgreSQL.

## Ecosistema Microsoft
SQL Server / Azure SQL.

## Entornos Oracle
Oracle Database.

## Analítica masiva en Google Cloud
BigQuery.

## MySQL/PostgreSQL administrado
Cloud SQL o AWS RDS.

## Data warehouse Cloud
Snowflake.



# 42. Ejercicio práctico

Con la base SQLite del notebook:

1. Consultar todos los clientes.
2. Filtrar mayores de 30 años.
3. Hacer `INNER JOIN`.
4. Calcular `utilidad = venta - costo`.
5. Calcular ventas por cliente.
6. Calcular ventas por forma de pago.
7. Encontrar el cliente con mayor venta.
8. Crear una CTE.
9. Crear un ranking con una función de ventana.
10. Guardar el resultado en una tabla SQL.



# 43. Conclusiones

CSV es solo una de muchas fuentes.

En proyectos reales es común combinar:

> **SQL + pandas + SQLAlchemy**

para trabajar directamente con:

- MySQL;
- PostgreSQL;
- SQL Server;
- Oracle;
- SQLite;
- DuckDB;
- BigQuery;
- Cloud SQL;
- AWS RDS;
- Azure SQL;
- Snowflake.

Después los resultados pueden utilizarse para:

- EDA;
- Machine Learning;
- automatización;
- dashboards;
- Data Engineering.



# 44. Referencias oficiales

- pandas SQL: https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html
- SQLAlchemy: https://docs.sqlalchemy.org/
- SQLite: https://docs.python.org/3/library/sqlite3.html
- PyMySQL: https://pymysql.readthedocs.io/
- PostgreSQL psycopg: https://www.psycopg.org/
- pyodbc: https://github.com/mkleehammer/pyodbc
- python-oracledb: https://python-oracledb.readthedocs.io/
- DuckDB: https://duckdb.org/docs/stable/clients/python/overview
- BigQuery Python: https://cloud.google.com/python/docs/reference/bigquery/latest
- Cloud SQL Python Connector: https://github.com/GoogleCloudPlatform/cloud-sql-python-connector
